In [ ]:
#importing packages
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [ ]:
#creating sparksessoins
spark = SparkSession.\
builder.\
appName("spark_SCD2").\
getOrCreate()

In [ ]:
customer_dim_data = [

(1,'manish','arwal','india','N','2022-09-15','2022-09-25'),
(2,'vikash','patna','india','Y','2023-08-12',None),
(3,'nikita','delhi','india','Y','2023-09-10',None),
(4,'rakesh','jaipur','india','Y','2023-06-10',None),
(5,'ayush','NY','USA','Y','2023-06-10',None),
(1,'manish','gurgaon','india','Y','2022-09-25',None),
]

customer_schema= ['id','name','city','country','active','effective_start_date','effective_end_date']

customer_dim_df = spark.createDataFrame(data= customer_dim_data,schema=customer_schema)

sales_data = [

(1,1,'manish','2023-01-16','gurgaon','india',380),
(77,1,'manish','2023-03-11','bangalore','india',300),
(12,3,'nikita','2023-09-20','delhi','india',127),
(54,4,'rakesh','2023-08-10','jaipur','india',321),
(65,5,'ayush','2023-09-07','mosco','russia',765),
(89,6,'rajat','2023-08-10','jaipur','india',321)
]

sales_schema = ['sales_id', 'customer_id','customer_name', 'sales_date', 'food_delivery_address','food_delivery_country', 'food_cost']

sales_df = spark.createDataFrame(data=sales_data,schema=sales_schema)

In [ ]:
customer_dim_df.show()

+---+------+-------+-------+------+--------------------+------------------+
| id|  name|   city|country|active|effective_start_date|effective_end_date|
+---+------+-------+-------+------+--------------------+------------------+
|  1|manish|  arwal|  india|     N|          2022-09-15|        2022-09-25|
|  2|vikash|  patna|  india|     Y|          2023-08-12|              NULL|
|  3|nikita|  delhi|  india|     Y|          2023-09-10|              NULL|
|  4|rakesh| jaipur|  india|     Y|          2023-06-10|              NULL|
|  5| ayush|     NY|    USA|     Y|          2023-06-10|              NULL|
|  1|manish|gurgaon|  india|     Y|          2022-09-25|              NULL|
+---+------+-------+-------+------+--------------------+------------------+



In [ ]:
sales_df.show()

+--------+-----------+-------------+----------+---------------------+---------------------+---------+
|sales_id|customer_id|customer_name|sales_date|food_delivery_address|food_delivery_country|food_cost|
+--------+-----------+-------------+----------+---------------------+---------------------+---------+
|       1|          1|       manish|2023-01-16|              gurgaon|                india|      380|
|      77|          1|       manish|2023-03-11|            bangalore|                india|      300|
|      12|          3|       nikita|2023-09-20|                delhi|                india|      127|
|      54|          4|       rakesh|2023-08-10|               jaipur|                india|      321|
|      65|          5|        ayush|2023-09-07|                mosco|               russia|      765|
|      89|          6|        rajat|2023-08-10|               jaipur|                india|      321|
+--------+-----------+-------------+----------+---------------------+-------------

**Join Both DF to indetify Changes**

In [ ]:
joined_data = customer_dim_df.join(sales_df,customer_dim_df["id"] == sales_df["customer_id"],"left")

In [ ]:
joined_data.show()

+---+------+-------+-------+------+--------------------+------------------+--------+-----------+-------------+----------+---------------------+---------------------+---------+
| id|  name|   city|country|active|effective_start_date|effective_end_date|sales_id|customer_id|customer_name|sales_date|food_delivery_address|food_delivery_country|food_cost|
+---+------+-------+-------+------+--------------------+------------------+--------+-----------+-------------+----------+---------------------+---------------------+---------+
|  1|manish|  arwal|  india|     N|          2022-09-15|        2022-09-25|      77|          1|       manish|2023-03-11|            bangalore|                india|      300|
|  1|manish|  arwal|  india|     N|          2022-09-15|        2022-09-25|       1|          1|       manish|2023-01-16|              gurgaon|                india|      380|
|  3|nikita|  delhi|  india|     Y|          2023-09-10|              NULL|      12|          3|       nikita|2023-09-20

In [ ]:
new_records_df = joined_data.where(
    (col("food_delivery_address") != col("city")) & (col("active") == "Y"))\
    .withColumn("effective_start_date",col("sales_date"))\
    .withColumn("effective_end_date",lit(None))\
      .select(
          "customer_id",
          "customer_name",
          col("food_delivery_address").alias("city"),
          col("food_delivery_country"),
          "active",
          "effective_start_date",
          "effective_end_date"
      )

new_records_df.show()

+-----------+-------------+---------+---------------------+------+--------------------+------------------+
|customer_id|customer_name|     city|food_delivery_country|active|effective_start_date|effective_end_date|
+-----------+-------------+---------+---------------------+------+--------------------+------------------+
|          1|       manish|bangalore|                india|     Y|          2023-03-11|              NULL|
|          5|        ayush|    mosco|               russia|     Y|          2023-09-07|              NULL|
+-----------+-------------+---------+---------------------+------+--------------------+------------------+



In [ ]:
old_records = joined_data.where(
        (col("food_delivery_address") != col("city")) & (col("active") == "Y"))\
        .withColumn("active", lit("N"))\
        .withColumn("effective_end_date", col("sales_date"))\
        .select(
            "customer_id",
            "customer_name",
            "city",
            "country",
            "active",
            "effective_start_date",
            "effective_end_date"
)
old_records.show()

+-----------+-------------+-------+-------+------+--------------------+------------------+
|customer_id|customer_name|   city|country|active|effective_start_date|effective_end_date|
+-----------+-------------+-------+-------+------+--------------------+------------------+
|          1|       manish|gurgaon|  india|     N|          2022-09-25|        2023-03-11|
|          5|        ayush|     NY|    USA|     N|          2023-06-10|        2023-09-07|
+-----------+-------------+-------+-------+------+--------------------+------------------+



**Find out new customer and insert them**

In [ ]:
new_customer = sales_df.join(
      customer_dim_df,sales_df["customer_id"] == customer_dim_df["id"],"leftanti")\
      .withColumn("active",lit("Y"))\
      .withColumn("effective_start_date",col("sales_date"))\
      .withColumn("effective_end_date",lit(None))\
      .select(
          "customer_id",
          "customer_name",
          "food_delivery_address",
          "food_delivery_country",
          "active",
          "effective_start_date",
          "effective_end_date"
      )
new_customer.show()

+-----------+-------------+---------------------+---------------------+------+--------------------+------------------+
|customer_id|customer_name|food_delivery_address|food_delivery_country|active|effective_start_date|effective_end_date|
+-----------+-------------+---------------------+---------------------+------+--------------------+------------------+
|          6|        rajat|               jaipur|                india|     Y|          2023-08-10|              NULL|
+-----------+-------------+---------------------+---------------------+------+--------------------+------------------+



**Merg all records in one df**

In [ ]:
final_records = customer_dim_df.union(new_records_df).union(old_records).union(new_customer)


final_records.orderBy(col('id'),col('effective_start_date').asc()).show()

+---+------+---------+-------+------+--------------------+------------------+
| id|  name|     city|country|active|effective_start_date|effective_end_date|
+---+------+---------+-------+------+--------------------+------------------+
|  1|manish|    arwal|  india|     N|          2022-09-15|        2022-09-25|
|  1|manish|  gurgaon|  india|     N|          2022-09-25|        2023-03-11|
|  1|manish|  gurgaon|  india|     Y|          2022-09-25|              NULL|
|  1|manish|bangalore|  india|     Y|          2023-03-11|              NULL|
|  2|vikash|    patna|  india|     Y|          2023-08-12|              NULL|
|  3|nikita|    delhi|  india|     Y|          2023-09-10|              NULL|
|  4|rakesh|   jaipur|  india|     Y|          2023-06-10|              NULL|
|  5| ayush|       NY|    USA|     N|          2023-06-10|        2023-09-07|
|  5| ayush|       NY|    USA|     Y|          2023-06-10|              NULL|
|  5| ayush|    mosco| russia|     Y|          2023-09-07|      

In [ ]:
window = Window.partitionBy("id","active").orderBy(col("effective_start_date").desc())
final_records.withColumn("rnk", rank().over(window)).show()

+---+------+---------+-------+------+--------------------+------------------+---+
| id|  name|     city|country|active|effective_start_date|effective_end_date|rnk|
+---+------+---------+-------+------+--------------------+------------------+---+
|  1|manish|  gurgaon|  india|     N|          2022-09-25|        2023-03-11|  1|
|  1|manish|    arwal|  india|     N|          2022-09-15|        2022-09-25|  2|
|  1|manish|bangalore|  india|     Y|          2023-03-11|              NULL|  1|
|  1|manish|  gurgaon|  india|     Y|          2022-09-25|              NULL|  2|
|  2|vikash|    patna|  india|     Y|          2023-08-12|              NULL|  1|
|  3|nikita|    delhi|  india|     Y|          2023-09-10|              NULL|  1|
|  4|rakesh|   jaipur|  india|     Y|          2023-06-10|              NULL|  1|
|  5| ayush|       NY|    USA|     N|          2023-06-10|        2023-09-07|  1|
|  5| ayush|    mosco| russia|     Y|          2023-09-07|              NULL|  1|
|  5| ayush|    

In [ ]:
window = Window.partitionBy("id","active").orderBy(col("effective_start_date").desc())
final_records.withColumn("rnk", rank().over(window))\
              .filter(~((col("active")=="Y")&(col("rnk")>=2))).show()

+---+------+---------+-------+------+--------------------+------------------+---+
| id|  name|     city|country|active|effective_start_date|effective_end_date|rnk|
+---+------+---------+-------+------+--------------------+------------------+---+
|  1|manish|  gurgaon|  india|     N|          2022-09-25|        2023-03-11|  1|
|  1|manish|    arwal|  india|     N|          2022-09-15|        2022-09-25|  2|
|  1|manish|bangalore|  india|     Y|          2023-03-11|              NULL|  1|
|  2|vikash|    patna|  india|     Y|          2023-08-12|              NULL|  1|
|  3|nikita|    delhi|  india|     Y|          2023-09-10|              NULL|  1|
|  4|rakesh|   jaipur|  india|     Y|          2023-06-10|              NULL|  1|
|  5| ayush|       NY|    USA|     N|          2023-06-10|        2023-09-07|  1|
|  5| ayush|    mosco| russia|     Y|          2023-09-07|              NULL|  1|
|  6| rajat|   jaipur|  india|     Y|          2023-08-10|              NULL|  1|
+---+------+----

In [ ]:
window = Window.partitionBy("id","active").orderBy(col("effective_start_date").desc())
final_records.withColumn("rnk", rank().over(window))\
              .filter(~((col("active")=="Y")&(col("rnk")>=2))).drop("rnk").show()

+---+------+---------+-------+------+--------------------+------------------+
| id|  name|     city|country|active|effective_start_date|effective_end_date|
+---+------+---------+-------+------+--------------------+------------------+
|  1|manish|  gurgaon|  india|     N|          2022-09-25|        2023-03-11|
|  1|manish|    arwal|  india|     N|          2022-09-15|        2022-09-25|
|  1|manish|bangalore|  india|     Y|          2023-03-11|              NULL|
|  2|vikash|    patna|  india|     Y|          2023-08-12|              NULL|
|  3|nikita|    delhi|  india|     Y|          2023-09-10|              NULL|
|  4|rakesh|   jaipur|  india|     Y|          2023-06-10|              NULL|
|  5| ayush|       NY|    USA|     N|          2023-06-10|        2023-09-07|
|  5| ayush|    mosco| russia|     Y|          2023-09-07|              NULL|
|  6| rajat|   jaipur|  india|     Y|          2023-08-10|              NULL|
+---+------+---------+-------+------+--------------------+------